In [0]:
df = spark.read.table("gizmobox.bronze.py_customers")
df.display()

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
df_filter = df.filter(col("customer_id").isNotNull())
df_filter.display()

In [0]:
df_distinct  = df_filter.dropDuplicates()
df_distinct.display()

In [0]:
df_max_ts = df_distinct.groupBy("customer_id").agg(max("created_timestamp").alias("max_ts"))
df_max_ts.display()

In [0]:
df_distinct_customer = df_distinct.join(df_max_ts,(df_distinct.customer_id == df_max_ts.customer_id) & (df_distinct.created_timestamp == df_max_ts.max_ts), "inner")\
    .select(df_distinct["*"])
df_distinct_customer.display()

In [0]:
df_casted_col  = df_distinct_customer.select(col("created_timestamp").cast("timestamp"),
                                             col("customer_id"),
                                             col("customer_name"),
                                             col("date_of_birth").cast("date"),
                                             col("email"),
                                             col("member_since").cast("date"),
                                             col("telephone")
                                             )
df_casted_col.display()

In [0]:
df_casted_col.writeTo("gizmobox.silver.py_customers").createOrReplace()

In [0]:
df = spark.table("gizmobox.silver.py_customers")
df.display()